In [1]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [2]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd

# fetch dataset
wholesale_customers = fetch_ucirepo(id=292)

# features
X = wholesale_customers.data.features.copy()

# Drop categorical columns
X = X.drop(columns=["Channel"])

# Convert to numpy
X = X.values

# Normalize
X = (X - X.mean(axis=0)) / X.std(axis=0)

print("Shape:", X.shape)

Shape: (440, 6)


# K-Means from Scratch

In [3]:
class KMeans:
    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters

    def initialize_centroids(self, X):
        indices = np.random.choice(len(X), self.k, replace=False)
        return X[indices]

    def assign_clusters(self, X, centroids):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        return np.argmin(distances, axis=1)

    def update_centroids(self, X, labels):
        centroids = []
        for i in range(self.k):
            cluster_points = X[labels == i]

            # Handle empty cluster
            if len(cluster_points) == 0:
                centroids.append(X[np.random.randint(0, len(X))])
            else:
                centroids.append(cluster_points.mean(axis=0))

        return np.array(centroids)

    def fit(self, X):
        self.centroids = self.initialize_centroids(X)

        for _ in range(self.max_iters):
            self.labels = self.assign_clusters(X, self.centroids)
            new_centroids = self.update_centroids(X, self.labels)

            if np.allclose(self.centroids, new_centroids):
                break

            self.centroids = new_centroids

        return self.labels, self.centroids

In [4]:
# run K-Means
k = 3

kmeans = KMeans(k=k)
labels, centroids = kmeans.fit(X)

print("K-Means centroids:\n", centroids)

K-Means centroids:
 [[-0.22932326 -0.23441579 -0.25581362 -0.17165943 -0.22710657 -0.15032227]
 [ 1.85656435  0.02767153 -0.17137337  1.40791752 -0.39387705  0.81874756]
 [-0.30641752  1.81191562  2.20636776 -0.25003895  2.23347101  0.25168469]]


# GMM from Scratch

In [5]:
class GMM:
    def __init__(self, k=3, max_iters=100, tol=1e-4):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol

    def multivariate_gaussian(self, X, mean, cov):
        d = X.shape[1]
        cov_inv = np.linalg.inv(cov)
        cov_det = np.linalg.det(cov)
        norm = 1.0 / np.sqrt((2 * np.pi) ** d * cov_det)
        diff = X - mean
        exp = np.exp(-0.5 * np.sum(diff @ cov_inv * diff, axis=1))
        return norm * exp

    def initialize(self, X, kmeans_centroids, kmeans_labels):
        self.means = kmeans_centroids.copy()
        self.weights = np.ones(self.k) / self.k

        d = X.shape[1]
        self.covariances = np.zeros((self.k, d, d))
        for i in range(self.k):
            cluster_points = X[kmeans_labels == i]
            if len(cluster_points) > 1:
                self.covariances[i] = np.cov(cluster_points.T)
            else:
                self.covariances[i] = np.eye(d)

    def e_step(self, X):
        # responsibility: P(cluster | point)
        n = X.shape[0]
        resp = np.zeros((n, self.k))
        for i in range(self.k):
            resp[:, i] = self.weights[i] * self.multivariate_gaussian(X, self.means[i], self.covariances[i])
        resp /= resp.sum(axis=1, keepdims=True)
        return resp

    def m_step(self, X, resp):
        n, d = X.shape
        Nk = resp.sum(axis=0)

        self.weights = Nk / n
        self.means = (resp.T @ X) / Nk[:, np.newaxis]

        for i in range(self.k):
            diff = X - self.means[i]
            self.covariances[i] = (resp[:, i][:, np.newaxis] * diff).T @ diff / Nk[i]
            # avoid singular covariance
            self.covariances[i] += np.eye(d) * 1e-6

    def log_likelihood(self, X):
        total = np.zeros(X.shape[0])
        for i in range(self.k):
            total += self.weights[i] * self.multivariate_gaussian(X, self.means[i], self.covariances[i])
        return np.sum(np.log(total))

    def fit(self, X, kmeans_centroids, kmeans_labels):
        self.initialize(X, kmeans_centroids, kmeans_labels)
        prev_ll = None

        for _ in range(self.max_iters):
            resp = self.e_step(X)
            self.m_step(X, resp)
            ll = self.log_likelihood(X)

            if prev_ll is not None and abs(ll - prev_ll) < self.tol:
                break
            prev_ll = ll

        self.labels = np.argmax(self.e_step(X), axis=1)
        return self.labels, self.means, self.covariances, self.weights

In [6]:
# run GMM initialized from K-Means
gmm = GMM(k=k)
gmm_labels, gmm_means, gmm_covs, gmm_weights = gmm.fit(X, centroids, labels)

print("GMM means:\n", gmm_means)
print("\nGMM mixing weights:\n", gmm_weights)

GMM means:
 [[-0.0249535  -0.39211299 -0.42618989 -0.1278095  -0.3869149  -0.19033803]
 [ 1.41404373  0.23242138 -0.12828853  1.69866744 -0.44742042  0.76602366]
 [-0.44231684  0.77599012  0.97728011 -0.31735855  1.00351792  0.14714179]]

GMM mixing weights:
 [0.61810658 0.09930281 0.28259061]


In [7]:
# compare K-Means vs GMM assignments
agreement = np.mean(labels == gmm_labels)
print(f"Cluster assignment agreement (raw): {agreement:.4f}")

print("\nK-Means cluster sizes:", [int((labels == i).sum()) for i in range(k)])
print("GMM cluster sizes:    ", [int((gmm_labels == i).sum()) for i in range(k)])

print(f"\nFinal GMM log-likelihood: {gmm.log_likelihood(X):.4f}")

Cluster assignment agreement (raw): 0.7636

K-Means cluster sizes: [346, 50, 44]
GMM cluster sizes:     [283, 40, 117]

Final GMM log-likelihood: -1738.8629
